# 03 — Data Preprocessing & Feature Engineering

This notebook implements the reproducible preprocessing workflow. The order is important: the train/test split happens **before fitting any imputer, scaler, or encoder**, preventing preprocessing leakage.

In [1]:
from pathlib import Path
import numpy as np
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "Crop Yield Data.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)

Project root: /tmp/cropsel/Crop Yield Prediction
Dataset: /tmp/cropsel/Crop Yield Prediction/data/Crop Yield Data.csv


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = pd.read_csv(DATA_PATH)
print("Initial shape:", df.shape)
print("Initial duplicates:", df.duplicated().sum())
print("Initial missing values:", int(df.isna().sum().sum()))

df = df.drop_duplicates().copy()
missing_target = int(df["Yield"].isna().sum())
df = df.dropna(subset=["Yield"]).reset_index(drop=True)
print("Removed duplicate rows:", int(df.duplicated().sum()))
print("Rows with missing target removed:", missing_target)
print("Shape after target cleaning:", df.shape)

Initial shape: (19689, 13)
Initial duplicates: 0
Initial missing values: 9
Removed duplicate rows: 0
Rows with missing target removed: 1
Shape after target cleaning: (19688, 13)


## Leakage control and feature engineering

`Production` is excluded because crop yield is directly derived from production and area in this type of dataset. Two input-only ratio features are added: fertilizer per unit area and pesticide per unit area. They use only predictors and are created before preprocessing.

In [3]:
X = df.drop(columns=["Yield","Production"]).copy()
y = df["Yield"].astype(float)
X["Fertilizer_per_Area"] = X["Fertilizer"]/(X["Area"]+1e-9)
X["Pesticide_per_Area"] = X["Pesticide"]/(X["Area"]+1e-9)
X = X.replace([np.inf,-np.inf],np.nan)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE)
print("Final feature columns:", list(X.columns))
print("Training shape:",X_train.shape,"Testing shape:",X_test.shape)
print("Target shapes:",y_train.shape,y_test.shape)

Final feature columns: ['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Avg_Temperature', 'Max_Temperature', 'Min_Temperature', 'Fertilizer_per_Area', 'Pesticide_per_Area']
Training shape: (15750, 13) Testing shape: (3938, 13)
Target shapes: (15750,) (3938,)


In [4]:
numeric_features=X.select_dtypes(include=np.number).columns.tolist()
categorical_features=X.select_dtypes(exclude=np.number).columns.tolist()
print("Numeric features:",numeric_features)
print("Categorical features:",categorical_features)

tree_preprocessor=ColumnTransformer([
 ("numeric",SimpleImputer(strategy="median"),numeric_features),
 ("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))]),categorical_features)
])
scaled_preprocessor=ColumnTransformer([
 ("numeric",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),numeric_features),
 ("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))]),categorical_features)
])
print("Preprocessors created without fitting on test data.")

Numeric features: ['Crop_Year', 'Area', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Avg_Temperature', 'Max_Temperature', 'Min_Temperature', 'Fertilizer_per_Area', 'Pesticide_per_Area']
Categorical features: ['Crop', 'Season', 'State']
Preprocessors created without fitting on test data.


In [5]:
# Fit only on training data — the critical leakage-prevention step.
X_train_processed=tree_preprocessor.fit_transform(X_train)
X_test_processed=tree_preprocessor.transform(X_test)
print("Processed train shape:",X_train_processed.shape)
print("Processed test shape:",X_test_processed.shape)
print("Test transformation used the training-fitted preprocessor: YES")
joblib.dump(tree_preprocessor,MODEL_DIR/"tree_preprocessor.pkl",compress=3)
joblib.dump(scaled_preprocessor,MODEL_DIR/"scaled_preprocessor_template.pkl",compress=3)

Processed train shape: (15750, 13)
Processed test shape: (3938, 13)
Test transformation used the training-fitted preprocessor: YES


['/tmp/cropsel/Crop Yield Prediction/models/scaled_preprocessor_template.pkl']

### Preprocessing result
- Duplicate rows are checked and removed.
- The one missing target row is removed rather than imputed.
- Missing predictor values are imputed **inside the pipeline**.
- Categorical values are encoded without looking at test categories during fitting.
- Scaling is applied only in the scaled pipeline used by support-vector modelling.
- `Production` is excluded to avoid target leakage.